### Supervised Learning Diagnostics
#### Lecture/Lab 24, Phys 4730 F25

<br>

Ref: V. Acquaviva, <i>Machine Learning for Physics and Astronomy</i> 

In [ ]:
%reset -f

In [ ]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

font = {'size'   : 14}
matplotlib.rc('font', **font)
matplotlib.rc('xtick', labelsize=14) 
matplotlib.rc('ytick', labelsize=14) 
matplotlib.rcParams['figure.dpi'] = 100

import sklearn.tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics 
from sklearn.model_selection import cross_val_predict, cross_val_score, cross_validate
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn import tree
from sklearn.tree import export_graphviz
from scipy import stats

import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)

### Read and tune the full HWC data set


In [ ]:
### read the dataset ###

df = pd.read_csv('20240321_hwc_full.csv', sep = ',')

In [ ]:
### Create a new data frame called bindf, dropping the ###
### old habitability label which had 3 possibilities.  ###
### Create instead a new habitability label which is   ###
### an OR of "optimistically habitable" and "expected  ###
### habitable". Recast the label as an integer.        ###

bindf = df.drop('P_HABITABLE', axis = 1) 
bindf['P_HABITABLE'] = (np.logical_or((df.P_HABITABLE == 1) , (df.P_HABITABLE == 2)))
bindf['P_HABITABLE'] = bindf['P_HABITABLE'].astype(int)

In [ ]:
### Confine ourself to 3(2) features and 1 target ###

NFEATURES = 2

if(NFEATURES == 3):
    final_features = bindf[['S_MASS', 'P_PERIOD', 'P_DISTANCE']]
else:
    final_features = bindf[['S_MASS', 'P_PERIOD']]
    
targets = bindf.P_HABITABLE   

In [ ]:
### get rid of any instance with at least one NaN in any column ###

final_features = final_features.dropna(axis = 0) 

In [ ]:
### Eliminates objects with > 5 sigma outliers in any column ###

final_features = final_features[(np.abs(stats.zscore(final_features)) < 5).all(axis=1)]

In [ ]:
### keep only targets corresponding to final features we have chosen ###

targets = targets[final_features.index]

In [ ]:
### reset the indices of the features and targets

final_features = final_features.reset_index(drop=True)
targets        = targets.reset_index(drop=True)

In [ ]:
### make a feature-space scatter plot of our data, in the familiar form ###

plt.figure(figsize=(10,6))

cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#20B2AA','#FF00FF'])

a = plt.scatter(final_features['S_MASS'], final_features['P_PERIOD'], \
                marker = '+', c=targets, s = 100, cmap=cmap)

plt.yscale('log')
plt.xlabel('Mass of Parent Star (Solar Mass Units)')
plt.ylabel('Period of Orbit (days)');

bluepatch    = mpatches.Patch(color='#20B2AA', label='Not Habitable')
magentapatch = mpatches.Patch(color='#FF00FF', label='Habitable')

plt.legend(handles=[magentapatch, bluepatch], loc = 'lower right', fontsize = 14);

plt.show()

In [ ]:
print("The fraction of positive targets is: %7.5f" % float(np.sum(targets)/len(targets)))

### Now, do the decision tree training

In [ ]:
### a random train/test split will assign 25% of instances to test set

Xtrain, Xtest, ytrain, ytest = train_test_split(final_features, targets, random_state=2)
Xtrain.shape, Xtest.shape

In [ ]:
### choose Decision Tree Method, build the model ###

model = DecisionTreeClassifier(random_state=5)
model.fit(Xtrain, ytrain)

In [ ]:
### check the accuracy scores ###

print("training set accuracy score:    %7.5f" % metrics.accuracy_score(ytrain, model.predict(Xtrain)))
print("testing set accuracy score:     %7.5f" % metrics.accuracy_score(ytest, model.predict(Xtest)))

In [ ]:
### sounds good, but suppose we had just classified everything as uninhabitable? ###

print("lazy classifier accuracy score: %7.5f" % metrics.accuracy_score(ytest, np.zeros(len(ytest))))

In [ ]:
### check precision and recall scores ###

print("precision score: %7.5f" % metrics.precision_score(ytest,model.predict(Xtest)))
print("recall score:    %7.5f" % metrics.recall_score(ytest,model.predict(Xtest)))

In [ ]:
### check confusion matrix ### 

cm = metrics.confusion_matrix(ytest,model.predict(Xtest))
print("confusion matrix: %10d  %10d" % (cm[0,0],cm[0,1])) 
print("                  %10d  %10d" % (cm[1,0],cm[1,1])) 

### Now, let's look at k-fold Cross Validation.

In [ ]:
### Version 1: Standard version. It doesn't shuffle the data,          ###
### so if your positive examples are all at the beginning or           ###
### all the end, it might lead to disastrous results.                  ### 

NS = 5

cv1 = KFold(n_splits = NS)

In [ ]:
### print out the class count for these splits ###

for train, test in cv1.split(final_features, targets):
    print('train -  {}   |   test -  {}'.format( \
    np.bincount(targets.loc[train]), np.bincount(targets.loc[test])))

In [ ]:
### use cross_validate function to find accuracy score ###

scores1 = cross_validate(DecisionTreeClassifier(), \
                         final_features, targets, cv = cv1, scoring = 'accuracy',
                         return_train_score = True)
scores1

In [ ]:
### compute average and standard deviation ###

print("The average accuracy is   {:.3f}".format(scores1['test_score'].mean()))
print("The standard deviation is {:.3f}".format(scores1['test_score'].std()))

In [ ]:
### use cross_validate function to find recall score ###

scores1 = cross_validate(DecisionTreeClassifier(random_state=1), \
                         final_features, targets, cv = cv1, scoring = 'recall',
                         return_train_score = True)
scores1

In [ ]:
### compute average and standard deviation ###

print("The average recall is     {:.3f}".format(scores1['test_score'].mean()))
print("The standard deviation is {:.3f}".format(scores1['test_score'].std()))

In [ ]:
### to get the confusion matrix, we need the predicted labels ###

model1 = DecisionTreeClassifier(random_state = 3)
y1     = cross_val_predict(model1, final_features, targets, cv = cv1) 

cm = metrics.confusion_matrix(targets,y1)
print("confusion matrix: %10d  %10d" % (cm[0,0],cm[0,1])) 
print("                  %10d  %10d" % (cm[1,0],cm[1,1])) 

### Finally, plot learning curves

A detailed description of the <b>learning_curve</b> function can be found [here](https://scikit-learn.org/stable/auto_examples/model_selection/plot_learning_curve.html).

Basically, we want to see how the scores vary with data set size. 

In [ ]:
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, title, X, y, ylim=None, cv=5, n_jobs=-1, \
                        train_sizes=np.linspace(.1, 1.0, 5), scoring = 'accuracy'):
    
    plt.figure(figsize=(10,6))
    plt.title(title)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.xlabel("Training examples")
    plt.ylabel(str(scoring))
    
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes, scoring = scoring)
    
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std  = np.std(train_scores, axis=1)
    test_scores_mean  = np.mean(test_scores, axis=1)
    test_scores_std   = np.std(test_scores, axis=1)
    plt.grid()

    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1,
                     color="r")

    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")

    plt.plot(train_sizes, train_scores_mean, 'o-', color="r",
             label="Training score")

    plt.plot(train_sizes, test_scores_mean,  'o-', color="g",
             label="Test score from cross-validation")

    plt.legend(loc="best")
    return plt

In [ ]:
model = DecisionTreeClassifier(random_state = 5)

In [ ]:
plot_learning_curve(model,'DT (default params)', final_features, targets, [0.0,1.1], \
                    cv = cv1, scoring = 'recall');

### Please answer the following questions:

* If the accuracy score is 100%, what will be the precision and recall scores?

* What is the effect of shuffling and stratification on the k-fold splits?

* Within the particular k-fold schemes, are the differences in scores statistically significant for any of the splits? Explain.

* Does the confusion matrix depend on the cross-validation scheme chosen? Why or why not?

* Do the learning curves exhibit any significant differences for the three cross-validation schemes?
